##O.W.L.E.T.-AI Performance Review

In [3]:
!pip install reportlab matplotlib pandas

import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.platypus import Paragraph, Frame, KeepInFrame, Spacer
from reportlab.lib.styles import ParagraphStyle
from google.colab import files

# =====================================================
# 1. DATA
# =====================================================

total_requests = 721
approvals = 497
strategic_redirections = 117
hallucinations_fixes = 67

gac_scores = {
    "Governance": 31,
    "Audit": 30,
    "Context": 39
}

metrics = {
    "Approvals": approvals,
    "Strategic Redirections": strategic_redirections,
    "Hallucinations/Fixes": hallucinations_fixes
}

# =====================================================
# 2. CREATE VISUALS
# =====================================================

plt.rcParams["font.family"] = "DejaVu Sans"

fig, ax = plt.subplots(figsize=(5.8, 2.55))
bars = ax.barh(
    list(metrics.keys()),
    list(metrics.values()),
    color=["#4A90E2", "#F5A623", "#D0021B"],
    height=0.55
)

ax.set_title("Project Log Outcomes", fontsize=13, fontweight="bold", pad=12, color="#2C3E50")
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.spines["left"].set_color("#DDDDDD")
ax.xaxis.set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=9, colors="#444444")

for bar in bars:
    ax.text(
        bar.get_width() + 12,
        bar.get_y() + bar.get_height() / 2,
        str(int(bar.get_width())),
        va="center",
        fontsize=9,
        fontweight="bold",
        color="#2C3E50"
    )

ax.set_xlim(0, max(metrics.values()) + 100)
plt.tight_layout()
bar_chart_path = "owlet_bar_chart.png"
plt.savefig(bar_chart_path, dpi=300, bbox_inches="tight", transparent=True)
plt.close()

fig, ax = plt.subplots(figsize=(3.8, 3.2))
sizes = list(gac_scores.values())
labels = list(gac_scores.keys())
pie_colors = ["#2ECC71", "#4A90E2", "#F1C40F"]

wedges, texts, autotexts = ax.pie(
    sizes,
    labels=None,
    autopct="%1.0f%%",
    startangle=90,
    pctdistance=0.72,
    colors=pie_colors,
    wedgeprops=dict(width=0.38, edgecolor="white", linewidth=2.5),
    textprops=dict(color="#2C3E50", fontsize=9, fontweight="bold")
)

ax.set_title("Managerial Balance", fontsize=13, fontweight="bold", pad=10, color="#2C3E50")
ax.legend(
    wedges,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    frameon=False,
    fontsize=8
)
ax.set_aspect("equal")
plt.tight_layout()
pie_chart_path = "owlet_donut_chart.png"
plt.savefig(pie_chart_path, dpi=300, bbox_inches="tight", transparent=True)
plt.close()

# =====================================================
# 3. PDF SETUP
# =====================================================

pdf_path = "OWLET_AI_Performance_Review_Final.pdf"

page_width, page_height = landscape(letter)
c = canvas.Canvas(pdf_path, pagesize=landscape(letter))

primary_dark = colors.HexColor("#2C3E50")
secondary_dark = colors.HexColor("#34495E")
bg_light = colors.HexColor("#F8F9FA")
border_color = colors.HexColor("#DEE2E6")
text_dark = colors.HexColor("#212529")
text_muted = colors.HexColor("#6C757D")

accent_green = colors.HexColor("#2ECC71")
accent_orange = colors.HexColor("#F39C12")
accent_red = colors.HexColor("#E74C3C")
accent_yellow_bg = colors.HexColor("#FFF8E1")
accent_yellow_border = colors.HexColor("#F5D76E")

section_style = ParagraphStyle(
    "Section",
    fontName="Helvetica-Bold",
    fontSize=8.6,
    leading=10,
    textColor=primary_dark,
    spaceAfter=3
)

body_style = ParagraphStyle(
    "Body",
    fontName="Helvetica",
    fontSize=6.6,
    leading=7.7,
    textColor=text_dark
)

small_style = ParagraphStyle(
    "Small",
    fontName="Helvetica",
    fontSize=6.2,
    leading=7,
    textColor=text_muted
)

# =====================================================
# 4. HELPER FUNCTIONS
# =====================================================

def draw_text_box(c, x, y, w, h, title, text, bg=colors.white, border=border_color):
    c.setFillColor(bg)
    c.setStrokeColor(border)
    c.roundRect(x, y, w, h, 8, fill=1, stroke=1)

    story = [
        Paragraph(title, section_style),
        Spacer(1, 2),
        Paragraph(text, body_style)
    ]

    padding_x = 12
    padding_y = 10

    frame = Frame(
        x + padding_x,
        y + padding_y,
        w - 2 * padding_x,
        h - 2 * padding_y,
        showBoundary=0
    )

    content = KeepInFrame(
        w - 2 * padding_x,
        h - 2 * padding_y,
        story,
        mode="shrink",
        hAlign="LEFT",
        vAlign="MIDDLE"
    )

    frame.addFromList([content], c)


def draw_metric_card(c, x, y, w, h, label, value, indicator_color):
    c.setFillColor(colors.white)
    c.setStrokeColor(border_color)
    c.roundRect(x, y, w, h, 8, fill=1, stroke=1)

    c.setFillColor(indicator_color)
    c.roundRect(x, y + h - 5, w, 5, 3, fill=1, stroke=0)

    c.setFillColor(primary_dark)
    c.setFont("Helvetica-Bold", 20)
    c.drawCentredString(x + w / 2, y + 23, str(value))

    c.setFillColor(text_muted)
    c.setFont("Helvetica-Bold", 6.8)
    c.drawCentredString(x + w / 2, y + 9, label.upper())

# =====================================================
# 5. HEADER
# =====================================================

c.setFillColor(primary_dark)
c.rect(0, page_height - 60, page_width, 60, fill=1, stroke=0)

c.setFont("Helvetica-Bold", 20)
c.setFillColor(colors.white)
c.drawString(32, page_height - 32, "O.W.L.E.T.-AI Performance Review")

c.setFont("Helvetica", 8.5)
c.setFillColor(colors.HexColor("#BDC3C7"))
c.drawString(32, page_height - 47, "AI Junior Assistant Evaluation Report | Generated using Python + Generative AI")

# =====================================================
# 6. METRIC CARDS
# =====================================================

margin = 30
gap = 12
card_w = (page_width - 2 * margin - 3 * gap) / 4
card_h = 52
top_y = page_height - 124

draw_metric_card(c, margin, top_y, card_w, card_h, "Total Requests", total_requests, secondary_dark)
draw_metric_card(c, margin + (card_w + gap), top_y, card_w, card_h, "Approvals", approvals, accent_green)
draw_metric_card(c, margin + 2 * (card_w + gap), top_y, card_w, card_h, "Strategic Redirections", strategic_redirections, accent_orange)
draw_metric_card(c, margin + 3 * (card_w + gap), top_y, card_w, card_h, "Hallucinations & Fixes", hallucinations_fixes, accent_red)

# =====================================================
# 7. FULL REPORT TEXT
# =====================================================

efficiency_text = """
O.W.L.E.T.-AI functioned as a junior assistant throughout my resistor recognition project. It showed strong value when used in structured tasks, especially in data preparation, where I adopted external code “snapshots” and needed to organize and adapt them into my workflow. In these cases, OWLET helped break down steps, restructure code, and accelerate implementation, which made it easier to integrate components into my project.<br/><br/>
However, its performance was not consistently reliable when tasks required technical interpretation, context awareness, or validation of outputs.<br/><br/>
Based on my project log, OWLET recorded <b>721 total requests</b>, <b>497 approvals</b>, <b>117 strategic redirections</b>, and <b>67 hallucinations/fixes</b>. These metrics suggest that while OWLET improved efficiency, it required continuous supervision. My managerial balance, which was analyzed in the <i>Final OWLET Performance Audit & Hiring Decision</i> discussion, was slightly pulled toward <b>Context (39%)</b>, followed by <b>Governance (31%)</b> and <b>Audit (30%)</b>. This reflects that a large portion of my effort was spent reminding OWLET of project details, restoring lost context, and verifying its outputs.<br/><br/>
This trend is directly supported by incidents such as OWLET repeatedly forgetting the status log and requiring reminders to restore tracking, as well as failing to detect uploaded files even after resending them. These examples highlight that OWLET struggled with continuity and context retention, requiring active managerial intervention.
"""

metrics_text = """
The project log metrics show that OWLET was effective when tasks were clearly defined and structured. The high number of approvals indicates that many outputs were useful in supporting the workflow. However, the <b>117 strategic redirections</b> demonstrate that I frequently had to override or guide the AI to align with technical requirements and project constraints.<br/><br/>
The <b>67 hallucinations/fixes</b> further reinforce that OWLET sometimes produced outputs that were confident but incorrect. For example, OWLET repeatedly generated code that failed during execution and required debugging, and in other cases it produced incorrect calculations in course-related tasks.<br/><br/>
Additionally, OWLET showed a tendency to oversimplify or overcomplicate solutions. In some cases, it presented all project ideas as feasible without considering time or scope constraints, while in others it introduced unnecessary complexity beyond project needs. These behaviors demonstrate that OWLET contributed to speed and structure, but not to independent technical reliability.
"""

strategic_text = """
A critical strategic redirection, which was also discussed in <i>Module 12: The OWLET Darwin Awards: Final Ethics Audit</i>, occurred during the data labeling phase of my specialist resistor model.<br/><br/>
To accelerate labeling in Label Studio, I used OWLET to interpret resistor band order instead of manually calculating each value. However, OWLET made a subtle but critical mistake by flipping the resistor orientation, specifically misplacing the gold tolerance band. This resulted in incorrect resistance values that still appeared realistic and internally consistent.<br/><br/>
Because the outputs looked valid, the issue was not immediately detected, allowing inaccurate labels to enter the dataset. This was a high-risk failure, as incorrect labels could propagate into model training, causing the YOLO model to learn incorrect mappings between color bands and resistance values. This would lead to false accuracy during testing and unreliable performance in real-world applications.<br/><br/>
This incident is directly reflected in the project log, where OWLET generated plausible but incorrect resistor values due to band misinterpretation.<br/><br/>
To address this, I overruled OWLET’s output and introduced a validation process before training. This included manually recalculating resistor values, verifying tolerance-band orientation, and sampling labeled data to ensure correctness. This strategic redirection prevented error propagation, preserved dataset integrity, and ensured the reliability of the final model.
"""

limitations_text = """
The project log metrics should be interpreted as approximate rather than exact values. OWLET sometimes lost track of the status log, and I did not consistently verify every classification of approvals, redirections, or hallucinations after each interaction. As a result, there are inconsistencies where the total number of requests does not perfectly align with the sum of categorized outcomes.<br/><br/>
This limitation is supported by incidents where OWLET forgot to maintain tracking or required repeated reminders to continue logging.<br/><br/>
This reflects a broader limitation of generative AI systems: while they can assist in organizing and accelerating workflows, they are not fully reliable as autonomous monitoring or tracking tools. Human oversight remains necessary to ensure both technical correctness and accurate performance evaluation.
"""

final_text = """
Overall, O.W.L.E.T.-AI proved to be valuable as a junior assistant, particularly in structured and repeatable tasks such as data preparation, workflow organization, and code adaptation. It significantly improved speed and productivity when used within clearly defined boundaries.<br/><br/>
However, it required continuous supervision in areas involving interpretation, decision-making, and validation. OWLET struggled with maintaining context, accurately interpreting ambiguous inputs, and consistently producing reliable outputs without oversight.<br/><br/>
From this experience, O.W.L.E.T.-AI is best understood as a support tool operating in a complex problem environment. It enhances efficiency, but engineering judgment, accountability, and decision-making remain the responsibility of the human manager. The success of the project depended not only on using AI, but on recognizing its limitations and strategically intervening when necessary.
"""

automation_text = """
This report was generated using Python to automate layout, integrate performance visuals, and produce a professional one-page PDF. Generative AI was used to support the analysis, structuring, and refinement of this performance evaluation.
"""

# =====================================================
# 8. VISUAL DASHBOARD
# =====================================================

chart_y = 314
chart_h = 145
visual_w = 420
callout_x = margin + visual_w + 14
callout_w = page_width - callout_x - margin

c.setFillColor(bg_light)
c.setStrokeColor(border_color)
c.roundRect(margin, chart_y, visual_w, chart_h, 8, fill=1, stroke=1)

c.drawImage(
    bar_chart_path,
    margin + 14,
    chart_y + 16,
    width=235,
    height=112,
    preserveAspectRatio=True,
    mask="auto"
)

c.drawImage(
    pie_chart_path,
    margin + 260,
    chart_y + 12,
    width=145,
    height=118,
    preserveAspectRatio=True,
    mask="auto"
)

draw_text_box(
    c,
    callout_x,
    chart_y,
    callout_w,
    chart_h,
    "Strategic Redirection: Critical Incident",
    strategic_text,
    bg=accent_yellow_bg,
    border=accent_yellow_border
)

# =====================================================
# 9. TEXT BOXES
# =====================================================

text_gap = 14
col_w = (page_width - 2 * margin - text_gap) / 2

upper_y = 184
upper_h = 118
lower_y = 42
lower_h = 130

draw_text_box(c, margin, upper_y, col_w, upper_h, "AI Efficiency Summary", efficiency_text)
draw_text_box(c, margin + col_w + text_gap, upper_y, col_w, upper_h, "Performance Metrics Interpretation", metrics_text)

draw_text_box(c, margin, lower_y, col_w, lower_h, "Data Reliability and Limitations", limitations_text)
draw_text_box(c, margin + col_w + text_gap, lower_y, col_w, lower_h, "Final Evaluation", final_text)

# =====================================================
# 10. FOOTER (DARK BAR STYLE - MATCH YOUR IMAGE)
# =====================================================

footer_height = 32

# Dark background bar
c.setFillColor(primary_dark)
c.rect(0, 0, page_width, footer_height, fill=1, stroke=0)

# Footer text (centered)
c.setFillColor(colors.white)
c.setFont("Helvetica", 8)

footer_text = "Automation Statement: This report was generated using Python to automate layout and visuals. Generative AI supported the analysis and refinement of this performance evaluation."

text_width = c.stringWidth(footer_text, "Helvetica", 8)

c.drawString(
    (page_width - text_width) / 2,   # center horizontally
    10,                              # vertical position
    footer_text
)

c.save()

print("PDF created successfully:", pdf_path)
files.download(pdf_path)

PDF created successfully: OWLET_AI_Performance_Review_Final.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>